# Gemini Iterative RAG Pipeline

1. Search the hybrid index (Gemini embeddings in Chroma + BM25, fused with reciprocal rank fusion) with the user's question.
2. Gemini reads the evidence so far and either says it is sufficient or writes targeted follow-up queries.
3. The follow-up queries are retrieved and fused with everything found earlier, for up to `rounds` rounds.
4. Gemini answers from the top fused chunks, with inline source ids.

In [ ]:
import importlib
import medx_retrieval

importlib.reload(medx_retrieval)
from medx_retrieval import answer_question, build_retriever, evaluate_questions, format_result

# The key is read from .env automatically.
bool(medx_retrieval.get_gemini_api_key())

In [ ]:
DATA_PATH = "Psichiatry_DSM-5.jsonl"

# First run embeds all chunks with Gemini (resumable if interrupted); later runs reuse chroma_db.
retriever = build_retriever(
    data_path=DATA_PATH,
    retriever_name="hybrid",
    chroma_path="chroma_db",
    collection_name="medx_dsm5",
    embedder="gemini",
)
retriever.count()

In [ ]:
response = answer_question(
    "How are panic attacks considered with anxiety disorders?",
    retriever,
    rounds=3,
    top_k=5,
    source_limit=6,
    refiner="gemini",
    generator="gemini",
)

print(response.answer)

In [ ]:
print("Stopped:", response.stop_reason)
for round_ in response.rounds:
    print(f"
Round {round_.round_number} ({round_.refiner})")
    if round_.missing_information:
        print("  missing:", round_.missing_information)
    for query in round_.queries:
        print("  query:", query)

print("
Sources:
")
for source in response.sources:
    print(format_result(source))
    print("-" * 80)

In [ ]:
# Batch retrieval + answer testing.
# refiner="keywords" and generator="extractive" avoid Gemini generation calls while tuning retrieval.
summary = evaluate_questions(
    "test_questions.jsonl",
    data_path=DATA_PATH,
    output_path="rag_test_results.jsonl",
    rounds=3,
    top_k=5,
    retriever="hybrid",
    embedder="gemini",
    refiner="gemini",
    generator="gemini",
)

summary